In [20]:
import pandas as pd
import sqlite3
import os
import numpy as np

# Load the dataset

In [21]:

# Note: header=None because the raw CSV doesn't have a header row
df = pd.read_csv('D:\\Healthcare_Query_Assistant\\healthcare_dataset.csv', header=None)

print(f"Initial shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows of raw data:")
print(df.head())



Initial shape: 55501 rows, 15 columns

First 5 rows of raw data:
              0    1       2           3                  4   \
0           Name  Age  Gender  Blood Type  Medical Condition   
1  Bobby JacksOn   30    Male          B-             Cancer   
2   LesLie TErRy   62    Male          A+            Obesity   
3    DaNnY sMitH   76  Female          A-            Obesity   
4   andrEw waTtS   28  Female          O+           Diabetes   

                  5                 6                           7   \
0  Date of Admission            Doctor                    Hospital   
1         2024-01-31     Matthew Smith             Sons and Miller   
2         2019-08-20   Samantha Davies                     Kim Inc   
3         2022-09-22  Tiffany Mitchell                    Cook PLC   
4         2020-11-18       Kevin Wells  Hernandez Rogers and Vang,   

                   8                   9            10              11  \
0  Insurance Provider      Billing Amount  Room Number 

# DATA CLEANING

In [22]:
# 1. Standardize column names for SQL (e.g., 'Blood Type' -> 'Blood_Type')
df.columns = df.iloc[0].astype(str).str.strip().str.replace(" ", "_", regex=False)
df = df.iloc[1:].reset_index(drop=True)

In [23]:
# 2. Fix messy name casing
df['Name'] = df['Name'].astype(str).str.title()

In [24]:
# 3. Ensure numeric columns are actually numeric
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Billing_Amount'] = pd.to_numeric(df['Billing_Amount'], errors='coerce')
df['Room_Number'] = pd.to_numeric(df['Room_Number'], errors='coerce')

In [25]:
# 4. Drop rows where critical data is missing
df.dropna(subset=['Name', 'Medical_Condition', 'Billing_Amount'], inplace=True)

In [28]:
# 5. Convert dates to datetime objects
df['Date_of_Admission'] = pd.to_datetime(df['Date_of_Admission'], errors='coerce')
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'], errors='coerce')

In [29]:
print(f"\nCleaned shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows of cleaned data:")
print(df.head())


Cleaned shape: 55500 rows, 15 columns

First 5 rows of cleaned data:
0           Name  Age  Gender Blood_Type Medical_Condition Date_of_Admission  \
0  Bobby Jackson   30    Male         B-            Cancer        2024-01-31   
1   Leslie Terry   62    Male         A+           Obesity        2019-08-20   
2    Danny Smith   76  Female         A-           Obesity        2022-09-22   
3   Andrew Watts   28  Female         O+          Diabetes        2020-11-18   
4  Adrienne Bell   43  Female        AB+            Cancer        2022-09-19   

0            Doctor                    Hospital Insurance_Provider  \
0     Matthew Smith             Sons and Miller         Blue Cross   
1   Samantha Davies                     Kim Inc           Medicare   
2  Tiffany Mitchell                    Cook PLC              Aetna   
3       Kevin Wells  Hernandez Rogers and Vang,           Medicare   
4    Kathleen Hanna                 White-White              Aetna   

0  Billing_Amount  Room_Numb

# Create SQLite Database & Indexes

In [31]:
db_path = '../healthcare.db'

# Remove old database if it exists to start fresh
if os.path.exists(db_path):
    os.remove(db_path)
    print("Removed existing database.")

In [32]:
# Connect to SQLite
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [33]:
# Save the cleaned dataframe to SQLite
df.to_sql('patients', conn, if_exists='replace', index=False)
print("Data saved to 'patients' table.")

Data saved to 'patients' table.


In [37]:
# Create indexes for efficient querying 
cursor.execute("CREATE INDEX IF NOT EXISTS idx_blood_type ON patients(Blood_Type);")
cursor.execute("CREATE INDEX idx_medical_condition ON patients(Medical_Condition);")
cursor.execute("CREATE INDEX idx_admission_date ON patients(Date_of_Admission);")
cursor.execute("CREATE INDEX idx_insurance ON patients(Insurance_Provider);")
conn.commit()
print("Indexes created successfully.")

OperationalError: index idx_medical_condition already exists

In [38]:
# Verify the data
query = "SELECT COUNT(*) as total, Blood_Type FROM patients GROUP BY Blood_Type ORDER BY total DESC;"
verification_df = pd.read_sql_query(query, conn)

In [39]:
conn.close()

print("\nDatabase Verification (Count by Blood Type):")
print(verification_df)
print("\n✅ Database 'healthcare.db' created successfully!")


Database Verification (Count by Blood Type):
   total Blood_Type
0   6969         A-
1   6956         A+
2   6947        AB+
3   6945        AB-
4   6945         B+
5   6944         B-
6   6917         O+
7   6877         O-

✅ Database 'healthcare.db' created successfully!
